In [1]:
# ANN has limitations 
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torchinfo import summary
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import torch.optim as optim


In [2]:
df=pd.read_csv('fmnist_small.csv')

In [3]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [4]:
# train test split
x=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [5]:
x


array([[  0,   0,   0, ..., 165,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       ...,
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0]], dtype=int64)

In [6]:
#splitting data into training and testing 
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [7]:
#scaling the data
X_train=X_train/255.0

X_test=X_test/255.0




In [8]:
# create CustomDataset Class
class dataset(Dataset):
    def __init__(self,features,labels):
        self.features= torch.tensor(features, dtype=torch.float32)
        self.labels= torch.tensor(labels, dtype=torch.float32)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):

        return self.features[index], self.labels[index]

In [9]:
# create train_dataset object
train_dataset =dataset(X_train, y_train)
test_dataset=dataset(X_test,y_test)
len(train_dataset)


4800

In [10]:
#creating dataloder object
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=32)

In [36]:
#creating a dataloder obj for traing and test
test_loader=DataLoader(test_dataset,shuffle=True,batch_size=32)

In [38]:
class my_cnn(nn.Module):
    def __init__(self,input_features):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(input_features,32,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(32,64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2,stride=2)
            )
        
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64,10),
            
            
        )
    def forward(self,x):
        out=self.features(x)
        out=self.classifier(out)
        return out
        
        

In [13]:
epochs=100
learning_rate=0.1

In [32]:
model=my_cnn(1)
loss_fn=nn.CrossEntropyLoss()
optimizer=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [34]:
#training loop 
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features,batch_labels in train_loader:
        batch_features = batch_features.view(-1, 1, 28, 28) 
        outputs=model(batch_features)
        #calculating loss
        loss=loss_fn(outputs,batch_labels.long())
        #back pass
        optimizer.zero_grad()
        loss.backward()
        ## update grads
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.080054340759913
Epoch: 2 , Loss: 0.6946021791299184
Epoch: 3 , Loss: 0.5837854670484861
Epoch: 4 , Loss: 0.541645913819472
Epoch: 5 , Loss: 0.4953239960471789
Epoch: 6 , Loss: 0.44513598511616387
Epoch: 7 , Loss: 0.42529602855443954
Epoch: 8 , Loss: 0.3703928086658319
Epoch: 9 , Loss: 0.3685576600333055
Epoch: 10 , Loss: 0.3253934251517057
Epoch: 11 , Loss: 0.29437888056350253
Epoch: 12 , Loss: 0.291221185028553
Epoch: 13 , Loss: 0.2767690821488698
Epoch: 14 , Loss: 0.23856488455086947
Epoch: 15 , Loss: 0.23512484724322955
Epoch: 16 , Loss: 0.2202049549172322
Epoch: 17 , Loss: 0.22166913449764253
Epoch: 18 , Loss: 0.19618874243150156
Epoch: 19 , Loss: 0.19704583369816342
Epoch: 20 , Loss: 0.1842444586008787
Epoch: 21 , Loss: 0.19103173242261012
Epoch: 22 , Loss: 0.15129077630117535
Epoch: 23 , Loss: 0.1345058040289829
Epoch: 24 , Loss: 0.13619474405422807
Epoch: 25 , Loss: 0.14707841358768442
Epoch: 26 , Loss: 0.13896431417825322
Epoch: 27 , Loss: 0.11018915360172589

In [48]:
#evaluation code with testr dat
total=0
correct=0
with torch.no_grad():
    for batch_features,batch_labels in test_loader:
        batch_features = batch_features.view(-1, 1, 28, 28)
        outputs=model(batch_features)
        _,predicted=torch.max(outputs,1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()
print(correct/total)

0.8541666666666666


In [46]:
#evaluation code with training data
total=0
correct=0
with torch.no_grad():
    for batch_features,batch_labels in train_loader:
        batch_features = batch_features.view(-1, 1, 28, 28)
        outputs=model(batch_features)
        _,predicted=torch.max(outputs,1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()
print(correct/total)

0.9922916666666667
